# Практическое занятие 6. Кластеризация режимов оборудования

Цель занятия - выполнить обучение без учителя для группировки
режимов оборудования по сенсорным признакам, сравнить k-means,
DBSCAN и Gaussian Mixture, а затем интерпретировать найденные
группы через диагностическую разметку.

Обучение без учителя (unsupervised learning) означает, что модель
не получает целевую переменную во время обучения. Диагностические
метки используются только после кластеризации для проверки
инженерной интерпретации.

## Инициализация среды выполнения

Ячейка ниже обеспечивает запуск блокнота в Google Colab и в
локальном Jupyter Notebook. Если проект уже открыт локально,
повторное клонирование не выполняется.

In [ ]:
# COLAB_BOOTSTRAP_APPailab
from pathlib import Path
import os
import sys

REQUIRED_PROCESSED_FILES = ['practice_06_equipment_modes_features.csv', 'practice_06_equipment_modes_diagnostics.csv']
PROJECT_REPOSITORY_URL = "https://github.com/Alexflex/appailab.git"

def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-colab.txt").exists() and (candidate / "data" / "processed").exists():
            return candidate
    return None

project_root = find_project_root(Path.cwd())

if project_root is None:
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB:
        workdir = Path("/content/appailab")
        if not workdir.exists():
            !git clone -q {PROJECT_REPOSITORY_URL} {workdir}
        project_root = workdir
        os.chdir(project_root)
        !pip install -q -r requirements-colab.txt
    else:
        raise FileNotFoundError(
            "Не найден корень проекта. Откройте блокнот из репозитория appailab "
            "или выполните git clone перед запуском."
        )

sys.path.insert(0, str(project_root / "src"))
missing_files = [
    name for name in REQUIRED_PROCESSED_FILES
    if not (project_root / "data" / "processed" / name).exists()
]
if missing_files:
    raise FileNotFoundError(
        "Не найдены подготовленные CSV: " + ", ".join(missing_files)
        + ". Выполните python scripts/generate_datasets.py."
    )

print(f"Корень проекта: {project_root}")
print("Проверенные CSV:", ", ".join(REQUIRED_PROCESSED_FILES))

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

DATA_DIR = project_root / "data" / "processed"
RANDOM_STATE = 20260507

## Теоретический блок

Кластеризация (clustering) - группировка объектов по сходству.
Метод k-средних (k-means) ищет центры кластеров и минимизирует
сумму квадратов расстояний до ближайшего центра.

DBSCAN (Density-Based Spatial Clustering of Applications with
Noise) - плотностной алгоритм, который выделяет плотные области и
помечает разреженные точки как шум. Gaussian Mixture Model (GMM),
гауссова смесь, описывает данные как смесь нескольких нормальных
распределений.

PCA (Principal Component Analysis), метод главных компонент,
используется для визуализации многомерных данных на плоскости.

In [ ]:
FEATURES_FILE = DATA_DIR / "practice_06_equipment_modes_features.csv"
DIAGNOSTICS_FILE = DATA_DIR / "practice_06_equipment_modes_diagnostics.csv"

df = pd.read_csv(FEATURES_FILE)
diagnostics_df = pd.read_csv(DIAGNOSTICS_FILE)
full_df = df.merge(diagnostics_df, on="sample_id", validate="one_to_one")

display(df.head())
print("Размер feature-таблицы:", df.shape)
print("Размер diagnostics-таблицы:", diagnostics_df.shape)

## Структура данных

В feature-CSV нет целевой переменной. Столбцы `true_mode_label`,
`mode_id`, `anomaly_flag` и `health_score` находятся только в
diagnostics-CSV. Их нельзя использовать в обучении кластеризации,
иначе задача обучения без учителя превращается в скрытую
классификацию.

In [ ]:
display(df.describe().T)
print("Пропуски:")
display(df.isna().sum().to_frame("missing_count"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["temperature_c"], bins=28, color="#4c78a8", edgecolor="white")
axes[0].set_title("Температура")
axes[0].set_xlabel("deg_C")

axes[1].hist(df["vibration_rms_mm_s"], bins=28, color="#e15759", edgecolor="white")
axes[1].set_title("Вибрация RMS")
axes[1].set_xlabel("mm/s")

axes[2].scatter(df["current_a"], df["temperature_c"], s=18, alpha=0.60)
axes[2].set_title("Ток и температура")
axes[2].set_xlabel("A")
axes[2].set_ylabel("deg_C")
plt.tight_layout()
plt.show()

## Выбор признаков и масштабирование

In [ ]:
# TODO: заполните список признаков. используйте только сенсорные признаки; не добавляйте true_mode_label и anomaly_flag
# Рекомендуемые признаки: ['speed_rpm', 'torque_nm', 'current_a', 'voltage_v', 'temperature_c', 'vibration_rms_mm_s', 'acoustic_db', 'cooling_flow_lpm', 'efficiency', 'pressure_kpa']
cluster_features = None
if cluster_features is None:
    raise ValueError('Заполните cluster_features: используйте только сенсорные признаки; не добавляйте true_mode_label и anomaly_flag')

forbidden_columns = {"true_mode_label", "mode_id", "anomaly_flag", "health_score", "maintenance_priority"}
leaked = forbidden_columns.intersection(cluster_features)
if leaked:
    raise ValueError(f"Обнаружена утечка диагностической разметки: {sorted(leaked)}")

X = df[cluster_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Масштабированная матрица признаков:", X_scaled.shape)

## PCA-визуализация

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
explained = pca.explained_variance_ratio_
print("Доля объясненной дисперсии PC1 и PC2:", np.round(explained, 4))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pca_df["PC1"], pca_df["PC2"], s=18, alpha=0.65)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Проекция данных на две главные компоненты")
plt.tight_layout()
plt.show()

## Выбор числа кластеров для k-means

In [ ]:
k_rows = []
for k in range(2, 8):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    k_rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
        }
    )
k_metrics_df = pd.DataFrame(k_rows)
display(k_metrics_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_metrics_df["k"], k_metrics_df["inertia"], marker="o")
axes[0].set_title("Метод локтя")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_metrics_df["k"], k_metrics_df["silhouette"], marker="o", color="#59a14f")
axes[1].set_title("Силуэтный коэффициент")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

## Обучение k-means, DBSCAN и Gaussian Mixture

In [ ]:
# TODO: задайте значение параметра. рекомендуемый диапазон числа кластеров: 3..6
# Рекомендуемое значение для первого запуска: 5
n_clusters = None
if n_clusters is None:
    raise ValueError('Заполните n_clusters: рекомендуемый диапазон числа кластеров: 3..6')
# TODO: задайте значение параметра. рекомендуемый диапазон eps для DBSCAN: 0.6..1.4
# Рекомендуемое значение для первого запуска: 0.95
dbscan_eps = None
if dbscan_eps is None:
    raise ValueError('Заполните dbscan_eps: рекомендуемый диапазон eps для DBSCAN: 0.6..1.4')

kmeans = KMeans(n_clusters=n_clusters, n_init=30, random_state=RANDOM_STATE)
kmeans_labels = kmeans.fit_predict(X_scaled)

dbscan = DBSCAN(eps=dbscan_eps, min_samples=8)
dbscan_labels = dbscan.fit_predict(X_scaled)

gmm = GaussianMixture(n_components=n_clusters, covariance_type="full", random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(X_scaled)

def safe_silhouette(labels: np.ndarray) -> float:
    unique = set(labels)
    if len(unique) <= 1 or len(unique) >= len(labels):
        return float("nan")
    return silhouette_score(X_scaled, labels)

clustering_metrics = pd.DataFrame(
    [
        {"method": "kmeans", "clusters": len(set(kmeans_labels)), "silhouette": safe_silhouette(kmeans_labels)},
        {"method": "dbscan", "clusters": len(set(dbscan_labels)) - int(-1 in set(dbscan_labels)), "silhouette": safe_silhouette(dbscan_labels)},
        {"method": "gaussian_mixture", "clusters": len(set(gmm_labels)), "silhouette": safe_silhouette(gmm_labels)},
    ]
)
display(clustering_metrics)

In [ ]:
cluster_plot_df = pca_df.copy()
cluster_plot_df["kmeans_cluster"] = kmeans_labels
cluster_plot_df["dbscan_cluster"] = dbscan_labels
cluster_plot_df["gmm_cluster"] = gmm_labels

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, column, title in zip(
    axes,
    ["kmeans_cluster", "dbscan_cluster", "gmm_cluster"],
    ["k-means", "DBSCAN", "Gaussian Mixture"],
):
    scatter = ax.scatter(cluster_plot_df["PC1"], cluster_plot_df["PC2"], c=cluster_plot_df[column], cmap="tab10", s=18, alpha=0.70)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

## Интерпретация кластеров

In [ ]:
interpreted_df = full_df.copy()
interpreted_df["kmeans_cluster"] = kmeans_labels

cluster_profile = interpreted_df.groupby("kmeans_cluster")[cluster_features].mean().round(3)
display(cluster_profile)

diagnostic_crosstab = pd.crosstab(
    interpreted_df["kmeans_cluster"],
    interpreted_df["true_mode_label"],
    normalize="index",
).round(3)
display(diagnostic_crosstab)

print("Adjusted Rand Index с диагностической разметкой:",
      round(adjusted_rand_score(interpreted_df["true_mode_label"], kmeans_labels), 4))

## Антипример: скрытая классификация вместо кластеризации

> **Внимание. АНТИПРИМЕР - НЕ ИСПОЛЬЗОВАТЬ КАК РАБОЧИЙ ПОДХОД.**
> Если добавить в признаки `mode_id`, `anomaly_flag` или
> `health_score`, алгоритм получает диагностическую разметку и
> перестает решать задачу обучения без учителя. Такой результат
> нельзя считать кластеризацией сенсорных режимов.

In [ ]:
leakage_features = cluster_features + ["mode_id", "anomaly_flag"]
leakage_df = full_df[leakage_features]
leakage_scaled = StandardScaler().fit_transform(leakage_df)
leakage_labels = KMeans(n_clusters=n_clusters, n_init=30, random_state=RANDOM_STATE).fit_predict(leakage_scaled)
print("ARI строгой кластеризации:",
      round(adjusted_rand_score(full_df["true_mode_label"], kmeans_labels), 4))
print("ARI антипримера с диагностической разметкой:",
      round(adjusted_rand_score(full_df["true_mode_label"], leakage_labels), 4))

## Задание для отчета

1. Объясните, чем кластеризация отличается от классификации.
2. Обоснуйте выбранное число кластеров по графикам `inertia` и
   силуэтного коэффициента.
3. Сравните k-means, DBSCAN и Gaussian Mixture.
4. Дайте инженерное описание 2-3 найденных кластеров.
5. Объясните, почему diagnostics-CSV нельзя использовать до
   завершения кластеризации.

Открытые источники для расширения: UCI AI4I 2020 Predictive
Maintenance Dataset и NASA C-MAPSS. Для открытых источников
необходимо отдельно фиксировать, какие столбцы являются
признаками, какие - отказами, а какие - служебной диагностической
разметкой.